# YgoMaster-PackBuilder — Install State / Update / Uninstall Tests

Test offline del ciclo completo:

```text
build
  ↓
install
  ↓
update
  ↓
uninstall
  ↓
stato originale ripristinato
```

Tutto viene eseguito in `TemporaryDirectory`; nessuna installazione reale di YgoMaster viene toccata.

Copertura principale:

- creazione dello stato locale;
- hash dei file installati;
- reinstallazione idempotente;
- update di una build;
- conservazione dello stato originale attraverso gli update;
- `PerPackRarities` originariamente assente / false / true;
- rimozione dei file creati da PackBuilder;
- backup e ripristino di file preesistenti;
- blocco su modifiche manuali;
- backup mancante;
- uninstall senza stato.


## 1. Import e fixture

In [1]:
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory
import json

from ygomaster_packbuilder.domain import (
    Card, OpeningPolicy, PackDefinition, PackType, ProjectDefinition,
    Rarity, RarityDistribution, SourcePackRef, SourceProvider,
    TenPackGuarantee, UltraRarePity,
)
from ygomaster_packbuilder.exporters.ygomaster import write_ygomaster_build
from ygomaster_packbuilder.installer import (
    YgoMasterInstallError,
    YgoMasterUninstallResult,
    install_ygomaster_build,
    uninstall_ygomaster_build,
)
from ygomaster_packbuilder.installer.state import (
    get_backup_path,
    get_install_state_dir,
    get_install_state_path,
    load_install_state,
    sha256_bytes,
)

def make_policy():
    return OpeningPolicy(
        key="configuration-a",
        normal_slot_distribution=RarityDistribution(55.0, 35.0, 7.5, 2.5),
        ten_pack_guarantee=TenPackGuarantee(
            True, RarityDistribution(0.0, 0.0, 80.0, 20.0)
        ),
        ultra_rare_pity=UltraRarePity(
            True, RarityDistribution(0.0, 0.0, 0.0, 100.0)
        ),
    )

def make_card(name, ydk_id, md_id, rarity):
    return Card(
        name=name,
        ydk_id=ydk_id,
        source_rarity=rarity.value,
        master_duel_id=md_id,
        rarity=rarity,
    )

def make_pack(key, name, description, pack_type, booster_size, cards):
    expected = {
        PackType.PERSONAL_MAIN: 100,
        PackType.PERSONAL_MINI: 20,
        PackType.SHARED_STAPLES: 200,
    }[pack_type]
    return PackDefinition(
        key=key,
        name=name,
        description=description,
        type=pack_type,
        source=SourcePackRef(
            provider=SourceProvider.YGOPRODECK,
            external_id=key,
        ),
        expected_card_count=expected,
        booster_size=booster_size,
        opening_policy_key="configuration-a",
        cards=cards,
    )

def make_project(version="v1"):
    return ProjectDefinition(
        name=f"Installer Lifecycle {version}",
        packs=[
            make_pack(
                "main-pack",
                f"Main Pack {version}",
                f"Main description {version}",
                PackType.PERSONAL_MAIN,
                7,
                [
                    make_card("Normal", 1, 1001, Rarity.N),
                    make_card("Ultra", 2, 1002, Rarity.UR),
                ],
            ),
            make_pack(
                "mini-pack",
                f"Mini Pack {version}",
                f"Mini description {version}",
                PackType.PERSONAL_MINI,
                3,
                [
                    make_card("Rare", 3, 1003, Rarity.R),
                    make_card("Super", 4, 1004, Rarity.SR),
                ],
            ),
        ],
        opening_policies=[make_policy()],
    )

pack_ids = {
    "main-pack": 9_000_000,
    "mini-pack": 9_000_001,
}

BASE_SHOP_ABSENT = '''{
    "DefaultSecretDuration": 86400,
    // preserve me
    "PackShop": {}
}
'''

BASE_SHOP_FALSE = '''{
    "PerPackRarities": false,
    "DefaultSecretDuration": 86400,
    "PackShop": {}
}
'''

BASE_SHOP_TRUE = '''{
    "PerPackRarities": true,
    "DefaultSecretDuration": 86400,
    "PackShop": {}
}
'''

def make_target(root, shop_text=BASE_SHOP_ABSENT):
    ygomaster_dir = Path(root) / "YgoMaster"
    data_dir = ygomaster_dir / "Data"
    data_dir.mkdir(parents=True)
    (data_dir / "Shop.json").write_bytes(shop_text.encode("utf-8"))
    return ygomaster_dir

def write_build(root, project):
    build_dir = Path(root) / "build"
    write_ygomaster_build(project, pack_ids, build_dir)
    return build_dir

print("Fixtures ready")

Fixtures ready


## 2. Install crea lo state corretto

In [2]:
with TemporaryDirectory() as tmp:
    project = make_project("v1")
    build_dir = write_build(tmp, project)
    ygomaster_dir = make_target(tmp)

    install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)

    state_path = get_install_state_path(ygomaster_dir)
    assert state_path.is_file()

    state = load_install_state(ygomaster_dir)
    assert state is not None
    assert state.version == 1
    assert state.per_pack_rarities_before is None

    assert set(state.managed_files) == {
        "Data/Shop/ygomaster-packbuilder.json",
        "Data/ShopPackOdds/ygomaster-packbuilder.json",
    }

    for managed in state.managed_files.values():
        target = ygomaster_dir / managed.relative_path
        assert managed.preexisting is False
        assert managed.installed_sha256 == sha256_bytes(target.read_bytes())

print("Initial state OK")

Initial state OK


## 3. Reinstall identico mantiene state e file

In [3]:
with TemporaryDirectory() as tmp:
    project = make_project("v1")
    build_dir = write_build(tmp, project)
    ygomaster_dir = make_target(tmp)

    install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)

    state_path = get_install_state_path(ygomaster_dir)
    before_state = state_path.read_bytes()

    result = install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)

    assert result.changed_files == ()
    assert state_path.read_bytes() == before_state

print("Identical reinstall OK")

Identical reinstall OK


## 4. Update cambia i file ma conserva l'origine

In [4]:
with TemporaryDirectory() as tmp:
    project_v1 = make_project("v1")
    build_dir = write_build(tmp, project_v1)
    ygomaster_dir = make_target(tmp, BASE_SHOP_FALSE)

    install_ygomaster_build(project_v1, pack_ids, build_dir, ygomaster_dir)

    state_v1 = load_install_state(ygomaster_dir)
    assert state_v1 is not None
    assert state_v1.per_pack_rarities_before is False

    old_hashes = {
        key: value.installed_sha256
        for key, value in state_v1.managed_files.items()
    }

    project_v2 = make_project("v2")
    write_ygomaster_build(project_v2, pack_ids, build_dir)

    result = install_ygomaster_build(project_v2, pack_ids, build_dir, ygomaster_dir)
    state_v2 = load_install_state(ygomaster_dir)

    assert state_v2 is not None
    assert state_v2.per_pack_rarities_before is False
    assert set(result.changed_files) == {
        ygomaster_dir / "Data/Shop/ygomaster-packbuilder.json",
    }

    new_hashes = {
        key: value.installed_sha256
        for key, value in state_v2.managed_files.items()
    }
    assert new_hashes != old_hashes

print("Update preserves original install state")

Update preserves original install state


## 5. Uninstall con proprietà originariamente assente

In [5]:
with TemporaryDirectory() as tmp:
    project = make_project("v1")
    build_dir = write_build(tmp, project)
    ygomaster_dir = make_target(tmp, BASE_SHOP_ABSENT)

    original_shop = (ygomaster_dir / "Data/Shop.json").read_bytes()

    install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    result = uninstall_ygomaster_build(ygomaster_dir)

    assert isinstance(result, YgoMasterUninstallResult)

    assert not (ygomaster_dir / "Data/Shop/ygomaster-packbuilder.json").exists()
    assert not (ygomaster_dir / "Data/ShopPackOdds/ygomaster-packbuilder.json").exists()
    assert not get_install_state_dir(ygomaster_dir).exists()

    assert (ygomaster_dir / "Data/Shop.json").read_bytes() == original_shop

    assert set(result.removed_files) == {
        ygomaster_dir / "Data/Shop/ygomaster-packbuilder.json",
        ygomaster_dir / "Data/ShopPackOdds/ygomaster-packbuilder.json",
    }

print("Absent property restored exactly")

Absent property restored exactly


## 6. Uninstall ripristina `false`

In [6]:
with TemporaryDirectory() as tmp:
    project = make_project("v1")
    build_dir = write_build(tmp, project)
    ygomaster_dir = make_target(tmp, BASE_SHOP_FALSE)

    original_shop = (ygomaster_dir / "Data/Shop.json").read_bytes()

    install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    uninstall_ygomaster_build(ygomaster_dir)

    assert (ygomaster_dir / "Data/Shop.json").read_bytes() == original_shop

print("false restored")

false restored


## 7. Se era già `true`, resta `true`

In [7]:
with TemporaryDirectory() as tmp:
    project = make_project("v1")
    build_dir = write_build(tmp, project)
    ygomaster_dir = make_target(tmp, BASE_SHOP_TRUE)

    original_shop = (ygomaster_dir / "Data/Shop.json").read_bytes()

    install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    state = load_install_state(ygomaster_dir)
    assert state is not None
    assert state.per_pack_rarities_before is True

    uninstall_ygomaster_build(ygomaster_dir)

    assert (ygomaster_dir / "Data/Shop.json").read_bytes() == original_shop

print("preexisting true preserved")

preexisting true preserved


## 8. File namespaced preesistenti vengono backuppati e ripristinati

In [8]:
with TemporaryDirectory() as tmp:
    project = make_project("v1")
    build_dir = write_build(tmp, project)
    ygomaster_dir = make_target(tmp)

    old_shop = b"OLD CUSTOM SHOP FILE\n"
    old_odds = b"OLD CUSTOM ODDS FILE\n"

    shop_target = ygomaster_dir / "Data/Shop/ygomaster-packbuilder.json"
    odds_target = ygomaster_dir / "Data/ShopPackOdds/ygomaster-packbuilder.json"
    shop_target.parent.mkdir(parents=True)
    odds_target.parent.mkdir(parents=True)
    shop_target.write_bytes(old_shop)
    odds_target.write_bytes(old_odds)

    install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)

    state = load_install_state(ygomaster_dir)
    assert state is not None
    assert all(item.preexisting for item in state.managed_files.values())

    assert get_backup_path(
        ygomaster_dir,
        "Data/Shop/ygomaster-packbuilder.json",
    ).read_bytes() == old_shop

    assert get_backup_path(
        ygomaster_dir,
        "Data/ShopPackOdds/ygomaster-packbuilder.json",
    ).read_bytes() == old_odds

    result = uninstall_ygomaster_build(ygomaster_dir)

    assert shop_target.read_bytes() == old_shop
    assert odds_target.read_bytes() == old_odds
    assert set(result.restored_files) >= {shop_target, odds_target}

print("Preexisting managed files restored")

Preexisting managed files restored


## 9. Update non sovrascrive il backup originale

In [9]:
with TemporaryDirectory() as tmp:
    project_v1 = make_project("v1")
    build_dir = write_build(tmp, project_v1)
    ygomaster_dir = make_target(tmp)

    target = ygomaster_dir / "Data/Shop/ygomaster-packbuilder.json"
    target.parent.mkdir(parents=True)
    original = b"ORIGINAL BEFORE PACKBUILDER\n"
    target.write_bytes(original)

    install_ygomaster_build(project_v1, pack_ids, build_dir, ygomaster_dir)

    backup = get_backup_path(
        ygomaster_dir,
        "Data/Shop/ygomaster-packbuilder.json",
    )
    assert backup.read_bytes() == original

    project_v2 = make_project("v2")
    write_ygomaster_build(project_v2, pack_ids, build_dir)
    install_ygomaster_build(project_v2, pack_ids, build_dir, ygomaster_dir)

    assert backup.read_bytes() == original

    uninstall_ygomaster_build(ygomaster_dir)
    assert target.read_bytes() == original

print("Original backup survives updates")

Original backup survives updates


## 10. File gestito modificato manualmente → update bloccato

In [10]:
with TemporaryDirectory() as tmp:
    project_v1 = make_project("v1")
    build_dir = write_build(tmp, project_v1)
    ygomaster_dir = make_target(tmp)

    install_ygomaster_build(project_v1, pack_ids, build_dir, ygomaster_dir)

    target = ygomaster_dir / "Data/Shop/ygomaster-packbuilder.json"
    target.write_text("MANUAL CHANGE", encoding="utf-8")

    project_v2 = make_project("v2")
    write_ygomaster_build(project_v2, pack_ids, build_dir)

    try:
        install_ygomaster_build(project_v2, pack_ids, build_dir, ygomaster_dir)
    except YgoMasterInstallError as exc:
        print("Expected:", exc)
    else:
        raise AssertionError("Manual modification should block update")

print("Manual modification blocks update")

Expected: Managed file was modified after installation: C:\Users\LMDP1\AppData\Local\Temp\tmpuw2ry3ym\YgoMaster\Data\Shop\ygomaster-packbuilder.json
Manual modification blocks update


## 11. File gestito modificato manualmente → uninstall bloccato

In [11]:
with TemporaryDirectory() as tmp:
    project = make_project("v1")
    build_dir = write_build(tmp, project)
    ygomaster_dir = make_target(tmp)

    install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)

    odds = ygomaster_dir / "Data/ShopPackOdds/ygomaster-packbuilder.json"
    odds.write_text("MANUAL CHANGE", encoding="utf-8")

    state_dir = get_install_state_dir(ygomaster_dir)
    main_shop_before = (ygomaster_dir / "Data/Shop.json").read_bytes()

    try:
        uninstall_ygomaster_build(ygomaster_dir)
    except YgoMasterInstallError as exc:
        print("Expected:", exc)
    else:
        raise AssertionError("Manual modification should block uninstall")

    assert state_dir.exists()
    assert (ygomaster_dir / "Data/Shop.json").read_bytes() == main_shop_before

print("Manual modification blocks uninstall before changes")

Expected: Managed file was modified after installation: C:\Users\LMDP1\AppData\Local\Temp\tmpav55fws8\YgoMaster\Data\ShopPackOdds\ygomaster-packbuilder.json
Manual modification blocks uninstall before changes


## 12. `PerPackRarities` modificato manualmente → uninstall bloccato

In [12]:
with TemporaryDirectory() as tmp:
    project = make_project("v1")
    build_dir = write_build(tmp, project)
    ygomaster_dir = make_target(tmp, BASE_SHOP_ABSENT)

    install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)

    shop_path = ygomaster_dir / "Data/Shop.json"
    text = shop_path.read_text(encoding="utf-8")
    shop_path.write_text(
        text.replace('"PerPackRarities": true', '"PerPackRarities": false'),
        encoding="utf-8",
    )

    try:
        uninstall_ygomaster_build(ygomaster_dir)
    except YgoMasterInstallError as exc:
        print("Expected:", exc)
    else:
        raise AssertionError("Manual Shop.json change should block restore")

    assert get_install_state_dir(ygomaster_dir).exists()
    assert (ygomaster_dir / "Data/Shop/ygomaster-packbuilder.json").exists()

print("Manual PerPackRarities change blocks uninstall")

Expected: PerPackRarities was manually changed after installation; refusing automatic restore.
Manual PerPackRarities change blocks uninstall


## 13. Backup richiesto ma mancante → uninstall bloccato

In [13]:
with TemporaryDirectory() as tmp:
    project = make_project("v1")
    build_dir = write_build(tmp, project)
    ygomaster_dir = make_target(tmp)

    target = ygomaster_dir / "Data/Shop/ygomaster-packbuilder.json"
    target.parent.mkdir(parents=True)
    target.write_bytes(b"PREEXISTING")

    install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)

    backup = get_backup_path(
        ygomaster_dir,
        "Data/Shop/ygomaster-packbuilder.json",
    )
    backup.unlink()

    try:
        uninstall_ygomaster_build(ygomaster_dir)
    except YgoMasterInstallError as exc:
        print("Expected:", exc)
    else:
        raise AssertionError("Missing backup should block uninstall")

    assert get_install_state_dir(ygomaster_dir).exists()

print("Missing backup detected before uninstall")

Expected: Missing installation backup: C:\Users\LMDP1\AppData\Local\Temp\tmpyaii6fc1\YgoMaster\.ygomaster-packbuilder\backup\Data\Shop\ygomaster-packbuilder.json
Missing backup detected before uninstall


## 14. Uninstall senza stato

In [14]:
with TemporaryDirectory() as tmp:
    ygomaster_dir = make_target(tmp)

    try:
        uninstall_ygomaster_build(ygomaster_dir)
    except YgoMasterInstallError as exc:
        print("Expected:", exc)
    else:
        raise AssertionError("Uninstall without state should fail")

print("Missing install state rejected")

Expected: No PackBuilder installation state was found.
Missing install state rejected
